# v4 有边界语义闭环修复 v1

日期：2026-11-02。源证据 `formal_30_v4_repair_scheduling_v1_results_20261102_224614`，283 文件校验通过。原结果为 2 完成、0 自动通过，保留原始产物、repair history 和评分，不重新解释为成功。

此前将旧产物通过推及为通用缺陷已解决不充分。本次区分 Prompt 约束、确定性修复边界、候选追溯及真实运行行为。仅复用开发样本，非独立测试。

## 实现与诊断更正

新增 `migration_pipeline/utils/bounded_semantics.cjs` 及 Python bridge：Babel AST 只解析，不执行生成代码；解析失败保持原样。

1. `repair_regression_guard.py`、`repair_scheduler.py`：用实际 Vue 源 `.self` 契约和父组件精确单分支包装体追溯调用。额外副作用、错目标、错参数、仅子组件同名方法不放行。
2. `stages/repair.py`：确定性计划和 LLM 返回后均应用有边界 AST 转换。只将无其它用途的纯算术箭头 computed formatter 转为普通方法；只对完整 SSM 源方法可证明为简单参数赋值／已有方法调用的事件处理器，同时还原源绑定与参数化方法体。不猜复杂源逻辑。
3. `repair_prompt.py`：补完整源事件方法和参数证据；明确禁止 this.event／data-* 绕行，修复绑定和签名必须配套；self-click 残留可进入聚焦提示。generation／服务器 Prompt 不变。

**更正前次诊断**：本地真实 San parseTemplate＋getEventListener 测试确认，裸 handler 默认注入 $event；handler() 才没有事件参数，handler($event) 为显式传参。先前将裸 handler 视为不传事件的说法不准确。此次不以这种错误假设拒收包装；显式传参仅作规范化。源事件业务参数丢失和 this.event 绕行仍是独立问题。

## 无模型离线回放

执行 `python -m migration_pipeline.replay_formal_30_v4_semantic_closure`。输入为冻结字节，输出单独写到 `data/experiments/analysis/formal_30_v4_semantic_closure_v1_20261102/`，原快照回放前后哈希不变。

| 输入 | 静态结果 | 初始 DOM 相似度 |
| --- | --- | ---: |
| TaskManager 原最终（第一次 LLM） | 仍四类错误，仍失败 | 0.9889 |
| TaskManager 原拒收（第二次 LLM） | 仍事件语法／computed 实例方法两类，仍失败 | 0.9889 |
| MonitoringDashboard 原最终 | 参数化 computed 及源事件参数错误由有界规则消除，静态通过 | 0.9605 |

TaskManager 第一份输入的合法 self 包装修复能通过候选追溯，不再因这一种追溯缺口被拒。它的另一处赋值事件 self guard 与计算依赖等仍待模型修复；原拒收候选也不是可直接认定成功的产物。

MonitoringDashboard 是离线规则回放通过，不是新模型迁移通过或人工交互通过。输出、输入哈希和每步错误／候选门见 replay.json；分析见同目录 REPORT.md。

## 验证、版本和待操作者运行

新增 `tests/test_bounded_semantic_repairs.py`，12 项正反例覆盖源契约、包装副作用／事件传参／组件作用域、事件方法恢复及拒修、computed 纯函数和隐藏依赖、解析失败、源证据 Prompt、真实 San 事件分发。算术方法以实际 San 实例比较 7／30／90 三个输入。完整测试、离线回放、dry-run 和哈希验证记录到新协议。

新批次 `formal_30_v4_semantic_closure_v1`，协议 `data/experiments/protocols/formal_30_v4_semantic_closure_v1_frozen_20261102/PROTOCOL.md`。最多 3 次 LLM repair，每次之前最多 4 个确定性步骤；qwen/qwen-plus，温度 0.2，600／660／660 秒超时，与前调度批相同预算。代码、Prompt 和生成随机性同时存在差异，不作单因素归因。

停止旧服务。窗口 A 设置环境 LLM_API_KEY 后运行 `python -m migration_pipeline.run_formal_30_v4_semantic_closure --server`；窗口 B 运行一次 `python -m migration_pipeline.run_formal_30_v4_semantic_closure --execute`。独立新输出目录，不加 --force，不自动调用模型。运行后先冻结再诊断。

此时尚无本批模型 summary，未修改测试页面、未直接手改原生成 San、未 commit／push。原 30 项历史评分保持，BudgetThreshold／TipSplitCalculator／LocalizationWorkspace 未结案；针对性回归后仍需完整 30 项回归，v5 探索性评估等待。

冻结前实际验证：完整 250/250 测试通过（含新增 12 项），耗时 48.650 秒；三份离线输入回放通过预定残留断言且无模型调用；dry-run 通过；新协议包含 262 份 payload、270 项 live 哈希校验通过。旧结果快照 283 项复核通过。

## 实际模型回归结果（2026-11-03）

运行 2026-11-03 14:15:45–14:51:57（UTC+8）。2/2 执行完成、0/2 自动通过、无执行错误；两项均达到 3 次 LLM repair 上限，无候选回退。初始 DOM 评估最终均通过，但没有新的自动交互评估或人工确认。

先冻结再详细分析：`data/experiments/snapshots/formal_30_v4_semantic_closure_v1_results_20261103_145253`，290 文件校验通过，冻结时协议 270 项 live 校验通过。生成 204,621、repair 554,944、总计 759,565 tokens。TaskManager 991.610 秒、MonitoringDashboard 1180.015 秒；这是样本总耗时，非各次请求耗时。

| 样本 | 确定性步骤（均接受） | LLM 调用 | 最终 DOM 相似度 | 静态残留 |
| --- | ---: | ---: | ---: | --- |
| TaskManager | 2 | 3 | 0.9889 | missing_data_types、invalid_san_computed_instance_method_call、hidden_template_reactive_dependency |
| MonitoringDashboard | 1 | 3 | 0.9605 | invalid_san_computed_instance_method_call、lifecycle_behavior_not_landed |

TaskManager：源 closeModal 的 self 包装可追溯，确定性包装和后续 onModalOverlayClick($event) 均未被误拒。第一次 LLM 后有 4 条／3 类错误，第二次后 2 类，第三次重新引入 getTaskCount 隐藏 tasks 依赖，变为 3 类。当前 gate 接受了这一退化，证明现有事件／关键编译／初始 DOM 保护没有覆盖全部语义错误复发。不能声称已有回退可防止所有旧问题再次出现；第二次候选也未通过，不回填成功。

TaskManager 三次 Prompt 分别为 444145／441376／440707 字符。missing_data_types 不在当前聚焦错误集合，后续仍走完整提示；这是可核对的路由问题，不能直接当作全部失败原因。父组件源 props 为空、输出整个脚本无 dataTypes，需结合内联子组件 props 审查契约，不通过添加空 dataTypes 或降标准刷分。

MonitoringDashboard：第一次 LLM 后处理实际运行 restored_source_parameterized_event_handler；最终不再报告事件参数、self guard、initData 命名冲突或 computed formatter 调用错误。剩余 getX/getY 是 computed 回调调组件实例方法，属于与纯算术箭头 formatter 不同的模式。生命周期方面，attached 的 initDataImpl 缺源 initData 最后的 loadSettings 调用，但设置加载逻辑已被移至 inited；需审查行为时序，不简单断言完全没有加载设置，也不直接豁免契约。

前次旧 MonitoringDashboard 产物离线回放通过，并不意味着本次重新生成能通过；新规则覆盖边界和本轮残留并不矛盾。与上一批预算上限相同，但实际 LLM 调用从 5 次变成 6 次，生成代码和 Prompt／规则也不同，不作等消耗或单因素因果比较。

证据与复现：`data/experiments/analysis/formal_30_v4_semantic_closure_v1_20261103_145253/REPORT.md`、summary.json；命令 `python -m migration_pipeline.analyze_formal_30_v4_semantic_closure --snapshot data/experiments/snapshots/formal_30_v4_semantic_closure_v1_results_20261103_145253`。

下一步建议（本次未实施）：为 TaskManager 第 2→3 次语义复发建立冻结回放和有边界拒收／候选保留；审查 dataTypes 与聚焦路由，处理 computed 共享纯计算及响应式依赖，按源契约恢复初始化时序。先离线验证再建新版本，不直接强制重抽旧批。原 30 项分数不改，v5 等待。本次仅新增冻结／分析工具和记录，未改运行期规则／Prompt／checker、原始 San 或测试页，未 commit／push。

## 后续开发版本（2026-11-03）

依据本批真实语义复发另建 `2026-11-03_formal_30_v4_semantic_closure_v2.ipynb`，新增具体语义拒收、opt-in residual_v2 计算 helper／源 props schema 规则及聚焦源生命周期证据。269 项本地测试、278 项新协议 live 哈希通过，尚未调用新模型批；原本批 0/2 结果和 290 文件原始快照不变。